# 01 — Análise exploratória (AED)
Previsão do segundo turno presidencial de 2026 (Lula × Flávio Bolsonaro) por município, com GLMs da família binomial.
Este caderno mostra a resposta, a relação com o turno anterior (defasagem), a linearidade das covariáveis na escala logit,
a colinearidade e a autocorrelação espacial, e termina com a lista de covariáveis recomendada.
Decisões e justificativas: `docs/pressupostos/02-aed.md`.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
import numpy as np, pandas as pd, geopandas as gpd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import graficos as gf

RAIZ = ".."
df = pd.read_parquet(f"{RAIZ}/data/processed/municipios.parquet").reset_index(drop=True)
df["log_pib_pc_2023"] = np.log(df["pib_per_capita_2023"])
for c in ["sigla_uf", "regiao", "nome_municipio"]:
    df[c] = df[c].astype(str)
gf.configurar(df, f"{RAIZ}/results/figures")
FIG = "aed_"

## Parte 1 — Dados
Fontes e colunas estão em `docs/dados/dicionario.md` e os cuidados de consolidação em `docs/dados/consolidacao.md`.
Cada linha é um município; as respostas são contagens binomiais (`votos_pt_2t_ano` de `n_2t_ano` votos nos dois candidatos).

In [2]:
print("Dimensões:", df.shape, "| municípios únicos:", df["id_municipio"].nunique())
reg = pd.read_csv(f"{RAIZ}/results/tables/resultado_regional.csv")
uf = pd.read_csv(f"{RAIZ}/results/tables/resultado_uf.csv")
nac = reg.groupby("ano")[["votos_pt", "votos_bolsonaro", "n"]].sum()
nac["prop_pt"] = nac["votos_pt"] / nac["n"]
display(nac.style.format({"votos_pt": "{:,.0f}", "votos_bolsonaro": "{:,.0f}", "n": "{:,.0f}", "prop_pt": "{:.2%}"}))
display(reg.pivot(index="regiao", columns="ano", values="prop_pt").style.format("{:.2%}"))

Dimensões: (5570, 161) | municípios únicos: 5570


,votos_pt,votos_bolsonaro,n,prop_pt
ano,,,,
2018,"46,987,176","57,666,176","104,653,352",44.90%
2022,"60,193,094","58,061,090","118,254,184",50.90%
2026,"64,055,133","59,652,191","123,707,324",51.78%


ano,2018,2022,2026
regiao,,,
Centro-Oeste,33.45%,39.79%,39.99%
Nordeste,69.69%,69.34%,69.37%
Norte,48.11%,48.97%,50.01%
Sudeste,34.63%,45.74%,47.24%
Sul,31.73%,38.16%,37.42%


In [3]:
tab_uf = uf.pivot(index="sigla_uf", columns="ano", values="prop_pt")
fig = go.Figure([go.Bar(x=tab_uf.index, y=tab_uf[a], name=str(a)) for a in tab_uf.columns])
fig.add_hline(y=0.5, line_dash="dash", line_color=gf.COR_REFERENCIA)
fig.update_yaxes(title_text="Proporção PT no 2º turno", tickformat=".0%")
fig.update_layout(barmode="group")
gf.mostrar_e_salvar(fig, FIG + "uf_resultado", titulo="Votação do PT por UF, 2º turno (2018, 2022, 2026)", altura=450)

In [4]:
print(df["n_2t_2026"].describe().round(0).to_string())
gf.histograma(np.log10(df["n_2t_2026"]), "log10(n_2t_2026)", name=FIG + "n_2t_2026",
              titulo="Votos nos dois candidatos por município (escala log10)")

count       5570.0
mean       22210.0
std       125318.0
min          848.0
25%         3815.0
50%         7201.0
75%        15331.0
max      6962277.0


A distribuição do tamanho é muito assimétrica (milhares de municípios pequenos e poucos milhões de votos nas capitais). Por isso a binomial,
que dá a cada município o peso de seu `n`, é preferível a um modelo gaussiano sobre a proporção, e as proporções dos municípios pequenos têm
muito mais ruído amostral que as das capitais.

## Parte 2 — Resposta
Proporção do PT no 2º turno de 2026: sem peso (cada município vale 1) e ponderada por votos (reproduz o resultado nacional).

In [5]:
gf.histograma(df["prop_pt_2t_2026"], "prop_pt_2t_2026", name=FIG + "hist_prop_sem_peso", titulo="prop_pt_2t_2026 (sem peso)")
gf.histograma(df["prop_pt_2t_2026"], "prop_pt_2t_2026", pesos=df["n_2t_2026"], name=FIG + "hist_prop_ponderado",
              titulo="prop_pt_2t_2026 (ponderado por votos)")
print("Média sem peso: %.4f | média ponderada (nacional): %.4f" % (
    df["prop_pt_2t_2026"].mean(), df["votos_pt_2t_2026"].sum() / df["n_2t_2026"].sum()))

Média sem peso: 0.5374 | média ponderada (nacional): 0.5178


A média sem peso fica abaixo da nacional porque os municípios pequenos e do Sul/Sudeste/Centro-Oeste pesam mais em contagem
que em votos. Isso mostra por que a agregação regional exige somar as contagens previstas, e não tirar a média das proporções.

Mapas (geometria simplificada em 0,12 grau; `matplotlib` não está no ambiente, então os mapas são Plotly).

In [6]:
geo = gpd.read_parquet(f"{RAIZ}/data/processed/municipios_geo.parquet")
geo["id_municipio"] = geo["id_municipio"].astype(df["id_municipio"].dtype)
geo = geo.set_index("id_municipio").loc[df["id_municipio"]].reset_index()
geo_s = geo.copy(); geo_s["geometry"] = geo.geometry.simplify(0.12)
hov = (df["nome_municipio"] + " (" + df["sigla_uf"] + ")").tolist()
gf.mapa(geo_s, df["prop_pt_2t_2026"], "Proporção PT, 2º turno 2026", FIG + "mapa_prop_2026", "prop_pt",
        centro=0.5, zmin=0, zmax=1, hover=hov)

In [7]:
lim = float(df["swing_logit_2026"].abs().quantile(0.99))
gf.mapa(geo_s, df["swing_logit_2026"], "Swing logit 2026 (logit 2026 menos logit 2022)", FIG + "mapa_swing_2026", "swing",
        centro=0, zmin=-lim, zmax=lim, hover=hov)

### Relação com o turno anterior
Se o voto de um município em um turno prediz o seguinte, a variável defasada `logit_pt_2t_2022` entra no modelo e o coeficiente deveria ser próximo de 1.
A inclinação OLS abaixo (sem peso e ponderada por `n`) mede isso na escala logit; o swing é a diferença em relação à reta de 45 graus.

In [8]:
def dispersao_lag(y, x, nome):
    d = df[[y, x, "n_2t_" + y[-4:], "regiao", "nome_municipio", "sigla_uf"]].copy()
    d.columns = ["y", "x", "n", "regiao", "nome_municipio", "sigla_uf"]
    ols = sm.OLS(d["y"], sm.add_constant(d["x"])).fit()
    wls = sm.WLS(d["y"], sm.add_constant(d["x"]), weights=d["n"]).fit()
    fig = go.Figure()
    for r, g in d.groupby("regiao"):
        fig.add_trace(go.Scattergl(x=g["x"], y=g["y"], mode="markers", name=r, marker=dict(size=4, opacity=0.5, color=gf.CORES_REGIAO[r]),
                       hovertext=g["nome_municipio"] + " (" + g["sigla_uf"] + ")",
                       hovertemplate="%{hovertext}<br>x=%{x:.2f}, y=%{y:.2f}<extra></extra>"))
    fig.add_trace(gf.curva_lowess(d["x"], d["y"], "LOWESS"))
    t = np.array([d["x"].min(), d["x"].max()])
    fig.add_trace(go.Scatter(x=t, y=ols.params["const"] + ols.params["x"] * t, mode="lines", name="OLS", line=dict(color="#0b0b0b", width=2)))
    fig.add_trace(go.Scatter(x=t, y=t, mode="lines", name="y = x", line=dict(color=gf.COR_REFERENCIA, dash="dash")))
    fig.update_xaxes(title_text=x); fig.update_yaxes(title_text=y)
    gf.mostrar_e_salvar(fig, FIG + nome, titulo=f"{y} x {x}", altura=520)
    return ols, wls

ols26, wls26 = dispersao_lag("logit_pt_2t_2026", "logit_pt_2t_2022", "lag_2026_2022")
print("2026 ~ 2022: inclinação OLS %.3f (R2 %.3f) | WLS por n %.3f (R2 %.3f)" % (ols26.params["x"], ols26.rsquared, wls26.params["x"], wls26.rsquared))

2026 ~ 2022: inclinação OLS 0.962 (R2 0.976) | WLS por n 0.929 (R2 0.969)


In [9]:
ols22, wls22 = dispersao_lag("logit_pt_2t_2022", "logit_pt_2t_2018", "lag_2022_2018")
print("2022 ~ 2018: inclinação OLS %.3f (R2 %.3f) | WLS por n %.3f (R2 %.3f)" % (ols22.params["x"], ols22.rsquared, wls22.params["x"], wls22.rsquared))

2022 ~ 2018: inclinação OLS 0.804 (R2 0.952) | WLS por n 0.753 (R2 0.940)


Leitura: inclinações próximas de 1 e R² altos indicam que o voto anterior explica quase toda a variação; a LOWESS por cima da reta mostra se a relação é linear
nas caudas e as cores mostram se há deslocamento por região (o swing regional). O que o backtest 2022 ← 2018 usa é exatamente esta relação.

## Parte 3 — Covariáveis
Candidatas (lista do projeto). `bloco_prefeito_2024` é categórica (esquerda/centro/direita/sem_prefeito) e `regiao` também; ambas entram nos modelos como fator.
Para dispersão, correlação e VIF, `bloco_prefeito_2024` vira duas indicadoras (esquerda e centro; base: direita e sem prefeito) e `pib_per_capita_2023` entra em log.

In [10]:
NUM = ["logit_pt_2t_2022", "nota_media_dep_2026", "pct_dep_esq_2026", "taxa_comparecimento_2t_2026", "delta_comparecimento_2t_2026",
       "log_populacao_2022", "log_densidade_2022", "log_eleitorado_2026", "pct_mulheres_2026", "pct_16_24_2026", "pct_60_mais_2026",
       "pct_superior_completo_2026", "pct_analfabeto_2026", "pct_ate_fund_incompleto_2026", "cobertura_pbf_2026", "log_pib_pc_2023",
       "pct_vab_agro_2021", "pct_evangelicos_2022", "pct_catolicos_2022", "pct_sem_religiao_2022", "pct_pretos_pardos_2022",
       "pct_indigenas_2022", "taxa_alfabetizacao_15mais_2022", "pct_urbana_2022"]
df["bloco_esq_2024"] = (df["bloco_prefeito_2024"] == "esquerda").astype(float)
df["bloco_centro_2024"] = (df["bloco_prefeito_2024"] == "centro").astype(float)
COV = NUM + ["bloco_esq_2024", "bloco_centro_2024"]
Y, N = df["votos_pt_2t_2026"], df["n_2t_2026"]
print("Valores ausentes nas candidatas:", int(df[COV].isna().sum().sum()))
# categóricas: logit empírico por nível
for c in ["bloco_prefeito_2024", "regiao"]:
    g = df.groupby(c).agg(y=("votos_pt_2t_2026", "sum"), n=("n_2t_2026", "sum"), municipios=("id_municipio", "size"))
    g["logit_empirico"] = np.log(g["y"] / (g["n"] - g["y"]))
    display(g[["municipios", "logit_empirico"]].round(3))

Valores ausentes nas candidatas: 0


,municipios,logit_empirico
bloco_prefeito_2024,,
centro,49,0.061
direita,4770,0.013
esquerda,735,0.547
sem_prefeito,16,-0.088


,municipios,logit_empirico
regiao,,
Centro-Oeste,467,-0.406
Nordeste,1794,0.817
Norte,450,0.0
Sudeste,1668,-0.11
Sul,1191,-0.514


### Linearidade na escala logit
Cada ponto é um decil da covariável: eixo x = média da covariável no decil; eixo y = logit empírico `log(ΣY / Σ(n−Y))`, que usa as contagens (municípios grandes pesam mais).
Se a curva é uma reta, o preditor linear é adequado; curvatura sugere transformação ou termo quadrático.
Atenção: `logit_pt_2t_2022` deve ficar quase de 45 graus, e as demais covariáveis refletem em boa parte a geografia (Nordeste × Sul).

In [11]:
def f_logit(v):
    g = gf.logit_empirico_por_decil(df[v], Y, N)
    return pd.DataFrame({"x": g["x"], "y": g["logit"]})
gf.painel_linearidade(COV, f_logit, "logit empírico 2026", FIG + "linearidade_logit", titulo="Logit empírico de 2026 por decil de cada covariável")

### Linearidade parcial: swing
O modelo contém a defasagem, então a relação relevante é a do swing (`swing_logit_2026` = logit 2026 − logit 2022) com cada covariável. Cada ponto é a média do swing no decil, ponderada por `n_2t_2026`.
`logit_pt_2t_2022` aparece aqui como regressão à média ou a efeito de inclinação diferente de 1.

In [12]:
def f_swing(v):
    g = gf.media_ponderada_por_decil(df[v], df["swing_logit_2026"], N)
    return pd.DataFrame({"x": g["x"], "y": g["z"]})
gf.painel_linearidade(COV, f_swing, "swing médio 2026", FIG + "linearidade_swing", titulo="Swing 2026 (média ponderada) por decil de cada covariável")

### Correlação de Spearman

In [13]:
corr = df[COV].corr(method="spearman")
gf.heatmap_correlacao(corr, "Correlação de Spearman entre as candidatas", FIG + "corr_spearman")
alta = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
print("Pares com |rho| >= 0.7:"); print(alta[alta.abs() >= 0.7].sort_values(key=abs, ascending=False).round(3).to_string())

Pares com |rho| >= 0.7:
log_populacao_2022            log_eleitorado_2026               0.992
pct_evangelicos_2022          pct_catolicos_2022               -0.924
pct_analfabeto_2026           taxa_alfabetizacao_15mais_2022   -0.895
pct_16_24_2026                pct_60_mais_2026                 -0.844
cobertura_pbf_2026            taxa_alfabetizacao_15mais_2022   -0.829
pct_superior_completo_2026    pct_ate_fund_incompleto_2026     -0.816
logit_pt_2t_2022              taxa_alfabetizacao_15mais_2022   -0.808
                              cobertura_pbf_2026                0.798
pct_16_24_2026                cobertura_pbf_2026                0.771
cobertura_pbf_2026            pct_pretos_pardos_2022            0.768
pct_ate_fund_incompleto_2026  taxa_alfabetizacao_15mais_2022   -0.764
pct_analfabeto_2026           pct_ate_fund_incompleto_2026      0.757
                              cobertura_pbf_2026                0.749
pct_catolicos_2022            pct_sem_religiao_2022            -0.

### Fator de inflação da variância (VIF)
Calculado com as covariáveis padronizadas e intercepto (VIF > 10: colinearidade severa; 5–10: atenção). Não inclui `regiao`.

In [14]:
def vif(cols):
    X = df[cols]; X = (X - X.mean()) / X.std(); X.insert(0, "const", 1.0)
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(cols))], index=cols, name="VIF")

vif_cheio = vif(COV)
DROP = {"pct_catolicos_2022": "composicional com evangélicos e sem religião (VIF 38)",
        "log_eleitorado_2026": "quase igual a log_populacao_2022 (r~1; VIF > 100)",
        "taxa_alfabetizacao_15mais_2022": "redundante com pct_analfabeto_2026 (mesmo constructo, mais recente e do eleitorado)",
        "nota_media_dep_2026": "redundante com pct_dep_esq_2026 (mesmo constructo, preferimos a fração de esquerda)"}
REDUZIDA = [c for c in COV if c not in DROP]
vif_red = vif(REDUZIDA)
tab = pd.concat([vif_cheio.rename("VIF_cheia"), vif_red.rename("VIF_reduzida")], axis=1)
tab["decisao"] = [("descartada: " + DROP[c]) if c in DROP else "mantida" for c in tab.index]
tab.round(2).to_csv(f"{RAIZ}/results/tables/vif_covariaveis.csv", index_label="covariavel")
display(tab.round(2))
print("VIF máximo: cheia %.1f (%s) | reduzida %.1f (%s)" % (vif_cheio.max(), vif_cheio.idxmax(), vif_red.max(), vif_red.idxmax()))

,VIF_cheia,VIF_reduzida,decisao
logit_pt_2t_2022,5.73,5.02,mantida
nota_media_dep_2026,2.91,NaN,descartada: redundante com pct_dep_esq_2026 (m...
pct_dep_esq_2026,2.40,1.23,mantida
taxa_comparecimento_2t_2026,1.86,1.76,mantida
delta_comparecimento_2t_2026,1.44,1.36,mantida
log_populacao_2022,124.17,2.51,mantida
log_densidade_2022,3.59,3.37,mantida
log_eleitorado_2026,121.37,NaN,descartada: quase igual a log_populacao_2022 (...
pct_mulheres_2026,3.40,3.19,mantida
pct_16_24_2026,7.33,6.87,mantida


VIF máximo: cheia 124.2 (log_populacao_2022) | reduzida 6.9 (pct_16_24_2026)


### Lista recomendada
Descartadas: `pct_catolicos_2022` (religiões somam ~1: as três juntas são quase singulares, mantemos evangélicos e sem religião),
`log_eleitorado_2026` (duplica `log_populacao_2022`), `taxa_alfabetizacao_15mais_2022` (duplica `pct_analfabeto_2026`) e
`nota_media_dep_2026` (duplica `pct_dep_esq_2026`).
Mantidas as demais; `pct_16_24_2026`, `pct_ate_fund_incompleto_2026`, `cobertura_pbf_2026`, `pct_60_mais_2026` e `logit_pt_2t_2022` ficam entre 5 e 7 de VIF: moderado,
mas aceitável; conferir estabilidade dos coeficientes no ajuste. Mais o fator `regiao`.

In [15]:
print("Recomendada (numéricas + indicadoras de bloco):"); print(", ".join(REDUZIDA), "+ regiao")

Recomendada (numéricas + indicadoras de bloco):
logit_pt_2t_2022, pct_dep_esq_2026, taxa_comparecimento_2t_2026, delta_comparecimento_2t_2026, log_populacao_2022, log_densidade_2022, pct_mulheres_2026, pct_16_24_2026, pct_60_mais_2026, pct_superior_completo_2026, pct_analfabeto_2026, pct_ate_fund_incompleto_2026, cobertura_pbf_2026, log_pib_pc_2023, pct_vab_agro_2021, pct_evangelicos_2022, pct_sem_religiao_2022, pct_pretos_pardos_2022, pct_indigenas_2022, pct_urbana_2022, bloco_esq_2024, bloco_centro_2024 + regiao


## Parte 4 — Autocorrelação espacial
Pesos: contiguidade Queen (função `pesos_queen`), ilhas ligadas ao vizinho mais próximo, padronização por linha. Moran's I global com 999 permutações.
A hipótese nula é de ausência de padrão espacial. Em dados de voto, esperamos forte autocorrelação da proporção; o swing, por já descontar o nível anterior, deve ter menos, mas ainda ter.

In [16]:
import esda
w = gf.pesos_queen(geo)
print("Municípios:", w.n, "| vizinhos médios: %.2f | ilhas após correção: %d" % (w.mean_neighbors, len(w.islands)))
res = {}
for v in ["prop_pt_2t_2026", "swing_logit_2026"]:
    np.random.seed(2026)
    mi = esda.Moran(df[v].values, w, permutations=999)
    res[v] = mi
    print(f"{v}: I = {mi.I:.4f} | E[I] = {mi.EI:.4f} | p (permutação) = {mi.p_sim:.3f} | z = {mi.z_sim:.1f}")

Municípios: 5570 | vizinhos médios: 5.79 | ilhas após correção: 0
prop_pt_2t_2026: I = 0.8958 | E[I] = -0.0002 | p (permutação) = 0.001 | z = 108.6


swing_logit_2026: I = 0.5422 | E[I] = -0.0002 | p (permutação) = 0.001 | z = 66.5


In [17]:
np.random.seed(2026)
lisa = esda.Moran_Local(df["swing_logit_2026"].values, w, permutations=999, seed=2026)
cod = np.where(lisa.p_sim < 0.05, lisa.q, 0)
nomes = {0: "Não significativo", 1: "Alto-Alto", 2: "Baixo-Alto", 3: "Baixo-Baixo", 4: "Alto-Baixo"}
print(pd.Series(cod).map(nomes).value_counts().to_string())
cores = [[0, "#d9d8d4"], [0.2, "#d9d8d4"], [0.2, "#d03b3b"], [0.4, "#d03b3b"], [0.4, "#e87ba4"], [0.6, "#e87ba4"],
         [0.6, "#2a78d6"], [0.8, "#2a78d6"], [0.8, "#6da7ec"], [1, "#6da7ec"]]
hov_l = [f"{h}: {nomes[c]}" for h, c in zip(hov, cod)]
gf.mapa(geo_s, cod, "LISA do swing logit 2026 (p < 0,05)", FIG + "mapa_lisa_swing", "cluster", escala=cores, zmin=-0.5, zmax=4.5,
        hover=hov_l, ticks=(list(nomes), list(nomes.values())))

Não significativo    3508
Alto-Alto            1045
Baixo-Baixo           844
Alto-Baixo             90
Baixo-Alto             83


Alto-Alto: municípios com swing alto cercados por vizinhos também altos (e Baixo-Baixo para o oposto). Se os agrupamentos seguem regiões, o fator `regiao` captura parte da dependência;
o que sobrar nos resíduos do GLM indica que é preciso um termo espacial ou erros-padrão robustos (verificar no caderno 02).